## Setup

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
import ast
import pandas as pd
import os

KAGGLE = True
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    print("Running on Kaggle")
else:
    KAGGLE = False
    print("Running Locally")

In [ ]:
model_path = '/kaggle/input/datasets/kamerondawson/bert-tiny-local/bert-tiny-local/'
MODEL_PATH = os.path.abspath(model_path)

## Data

In [ ]:
def max_min_fair_truncation(tok_p, tok_a, tok_b, max_length):
    """
    Distributes capacity fairly among prompt and two responses.
    Uses max-min fairness: satisfy shortest sequence first.
    """
    toks = [('p', tok_p), ('a', tok_a), ('b', tok_b)]
    # Sort by length ascending to satisfy shortest first
    toks.sort(key=lambda x: len(x[1]))
    
    # RoBERTa uses <s> Prompt </s> RespA </s> RespB </s>
    # That is 1 CLS and 3 SEP tokens = 4 special tokens total
    capacity = max_length - 4
    remaining_items = len(toks)
    
    results = {}
    for tid, tok in toks:
        # Fair share is the remaining capacity divided by remaining items
        fair_share = capacity // remaining_items
        alloc = min(len(tok), fair_share)
        
        results[tid] = tok[:alloc]
        capacity -= len(results[tid])
        remaining_items -= 1
        
    return results['p'], results['a'], results['b']

def max_min_fair_head_tail_truncation(tok_p, tok_a, tok_b, max_length):
    """
    Uses max-min fairness but splits between head and tail of response rather than just the head.
    """
    toks = [('p', tok_p), ('a', tok_a), ('b', tok_b)]
    toks.sort(key=lambda x: len(x[1]))
    capacity = max_length - 4
    remaining_items = len(toks)
    results = {}
    for tid, tok in toks:
        fair_share = capacity // remaining_items
        alloc = min(len(tok), fair_share)
        results[tid] = tok[:alloc//2] + tok[-alloc//2:] # Only difference is this
        capacity -= len(results[tid])
        remaining_items -= 1
    return results['p'], results['a'], results['b']

In [ ]:
def fix_str(string):
    if string.startswith('["') or string.startswith('[\''):
        string = string[2:]
    elif string.startswith('[') or string.startswith('\''):
        string = string[1:]
    if string.endswith('"]') or string.endswith('\']'):
        string = string[:-2]
    elif string.endswith(']') or string.endswith('\''):
        string = string[:-1]
    return string

class LlmPreferenceDataset(Dataset):
    def __init__(self, dataframe, is_test=False, model_name=MODEL_PATH, max_length=512):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
        self.max_length = max_length
        self.is_test = is_test
        
        if not self.is_test:
            # Create new samples by swapping each response to remove positional bias
            orig_features = dataframe[['prompt', 'response_a', 'response_b']].values
            orig_labels = dataframe[['winner_model_a', 'winner_model_b', 'winner_tie']].values
            
            swapped_features = orig_features.copy()
            swapped_features[:, [1, 2]] = swapped_features[:, [2, 1]] # Swap Resp_A and Resp_B
            
            swapped_labels = orig_labels.copy()
            swapped_labels[:, [0, 1]] = swapped_labels[:, [1, 0]] # Swap Label_A and Label_B
            
            self.features = np.vstack((orig_features, swapped_features))
            self.labels = np.vstack((orig_labels, swapped_labels))
        else:
            self.features = dataframe[['prompt', 'response_a', 'response_b']].values
        
    def __len__(self):
        return len(self.features)

    def embed_sample(self, idx, reverse=False):
        # Extract raw data
        raw_prompt, raw_resp_a, raw_resp_b = self.features[idx]
        if reverse:
            tmp = raw_resp_a
            raw_resp_a = raw_resp_b
            raw_resp_b = tmp
        # Fix formatting
        prompt, resp_a, resp_b = fix_str(raw_prompt), fix_str(raw_resp_a), fix_str(raw_resp_b)
        
        # Tokenize raw text (without special tokens)
        tok_p = self.tokenizer.tokenize(str(prompt))
        tok_a = self.tokenizer.tokenize(str(resp_a))
        tok_b = self.tokenizer.tokenize(str(resp_b))
        
        # Apply custom truncation
        tok_p, tok_a, tok_b = max_min_fair_head_tail_truncation(tok_p, tok_a, tok_b, self.max_length)
        
        # Construct the token sequence manually
        # RoBERTa format: <s> A </s> B </s> C </s>
        # Note: RoBERTa uses SINGLE </s> between segments
        sep = self.tokenizer.sep_token
        cls = self.tokenizer.cls_token
        tokens = [cls] + tok_p + [sep] + tok_a + [sep] + tok_b + [sep]
        
        # Convert to IDs
        input_ids = self.tokenizer.convert_tokens_to_ids(tokens)
        
        # Pad to max_length
        padding_length = self.max_length - len(input_ids)
        attention_mask = [1] * len(input_ids) + [0] * padding_length
        input_ids = input_ids + [self.tokenizer.pad_token_id] * padding_length

        return input_ids, attention_mask
    
    def __getitem__(self, idx):
        input_ids, attention_mask = self.embed_sample(idx)
        
        sample = {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
        }
        if not self.is_test:
            sample['label'] = torch.tensor(self.labels[idx], dtype=torch.float)
        else:
            # Get swapped version as well
            input_ids_swapped, attention_mask_swapped = self.embed_sample(idx, reverse=True)
            sample['input_ids_swapped'] = torch.tensor(input_ids_swapped, dtype=torch.long)
            sample['attention_mask_swapped'] = torch.tensor(attention_mask_swapped, dtype=torch.long)
            
        return sample

In [ ]:
if KAGGLE:
    train_set_filepath = '/kaggle/input/llm-classification-finetuning/train.csv'
    test_set_filepath = '/kaggle/input/llm-classification-finetuning/test.csv'
else:
    train_set_filepath = os.path.join(os.getcwd(), 'train.csv')
    test_set_filepath = os.path.join(os.getcwd(), 'test.csv')

# Load datasets
base_train_df = pd.read_csv(train_set_filepath)
if os.path.exists(test_set_filepath):
    base_test_df = pd.read_csv(test_set_filepath)
else:
    base_test_df = None
base_train_df.dropna()
base_test_df.dropna()

# Only the train set has labels so we need to split that into our real train and test sets
train_df, val_df = train_test_split(base_train_df, test_size=0.2, random_state=42)

# Convert to torch datasets
train_set = LlmPreferenceDataset(train_df, is_test=False)
val_set = LlmPreferenceDataset(val_df, is_test=False)
if base_test_df is not None:
    test_set = LlmPreferenceDataset(base_test_df, is_test=True)
else:
    test_set = None

# Create data loaders
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False)
if test_set is not None:
    test_loader = DataLoader(test_set, batch_size=32, shuffle=False)
else:
    test_loader = None

In [ ]:
# Verify truncation worked
# for sample in train_set:
#     if len(sample['input_ids']) > train_set.max_length:
#         raise Exception(f'>{train_set.max_length} tokens')

In [ ]:
# Test with a sample
sample = train_set[0]
print(f"Length: {len(sample['input_ids'])}")
print(f"Decoded: {train_set.tokenizer.decode(sample['input_ids'])}")

In [ ]:
# Get an idea of the size of the inputs
lengths = []
for row in base_train_df.itertuples(index=False):
    prompt, resp_a, resp_b = row[3:6]
    length = len(prompt) + len(resp_a) + len(resp_b)
    lengths.append(length)
lengths.sort()
print('Min, median, max input lengths: ', lengths[0], lengths[len(lengths)//2], lengths[-1])

## Model

In [ ]:
class LlmPreferenceModel(nn.Module):
    def __init__(self, model_name=MODEL_PATH, num_classes=3):
        super(LlmPreferenceModel, self).__init__()
        self.transformer = AutoModel.from_pretrained(model_name, local_files_only=True)
        hidden_size = self.transformer.config.hidden_size

        # Classification head
        # Dropout layer for regularization to prevent overfitting
        self.dropout = nn.Dropout(0.3) 
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state
        cls_output = last_hidden_state[:, 0, :]

        # Apply dropout and linear layer
        pooled_output = self.dropout(cls_output)
        logits = self.classifier(pooled_output)
        
        return logits

In [ ]:
DEVICE = torch.device('cpu')
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
print(f"Using device: {DEVICE}")

criterion = nn.CrossEntropyLoss()

# TODO Mix architectures for maximum diversity
architectures = [LlmPreferenceModel] 
models = []
optimizers = []
schedulers = []

for i, ArchClass in enumerate(architectures):
    torch.manual_seed(42 + i)
    model = ArchClass().to(DEVICE)
    models.append(model)
    # optimizer = optim.Adam(model.parameters(), lr=0.0001)
    optimizer = optim.AdamW(model.parameters(), lr=0.00001, weight_decay=0.01)
    optimizers.append(optimizer)
    schedulers.append(optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=1))

# Track accuracy for weighting the final vote
model_scores = [float('inf')] * len(models)

## Train

In [ ]:
# Validation
def test_model(model):
    model.eval() # Set model to evaluation mode
    total_log_loss = 0
    total_val_loss = 0.0
    with torch.no_grad():
        for data in val_loader:
            # Move data to device
            input_ids = data['input_ids'].to(DEVICE)
            attention_mask = data['attention_mask'].to(DEVICE)
            targets = data['label'].to(DEVICE)
            
            # Run model
            logits = model(input_ids, attention_mask)

            # Update loss
            loss = criterion(logits, targets) # Calculate loss
            total_val_loss += loss.item()

            # Update score
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            target_np = targets.cpu().numpy()
            target_indices = np.argmax(target_np, axis=1) 
            total_log_loss += log_loss(target_indices, probs, labels=[0, 1, 2])

    avg_val_loss = total_val_loss / len(val_loader)
    avg_log_loss = total_log_loss / len(val_loader)
    print(f'Leaderboard Log Loss: {avg_log_loss:.4f} | Val Loss: {avg_val_loss:.4f}')
    return avg_val_loss, avg_log_loss

In [ ]:
# Train
EPOCHS = 5
for m_idx, model in enumerate(models):
    print(f"Training Model {m_idx + 1}/{len(models)}...")
    optimizer = optimizers[m_idx]
    scheduler = schedulers[m_idx]

    best_val_loss = float('inf')
    
    for epoch in range(EPOCHS):
        model.train() # Set model back to training mode
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            input_ids = data['input_ids'].to(DEVICE)
            attention_mask = data['attention_mask'].to(DEVICE)
            targets = data['label'].to(DEVICE)
            
            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
            if i % 100 == 99: # Print every 100 batches
                print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.3f}')
                running_loss = 0.0
    
        # Run validation and get the loss
        current_val_loss, current_log_loss = test_model(model)
        model_scores[m_idx] = min(model_scores[m_idx], current_log_loss)

        # Save the best weights (Manual Early Stopping)
        if current_val_loss < best_val_loss:
            best_val_loss = current_val_loss
            torch.save(model.state_dict(), 'best_model_weights.pth')
            print(f'New best model saved at Epoch {epoch+1}')
        
        # Step the scheduler based on the validation loss
        scheduler.step(current_val_loss)
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f'End of Epoch {epoch + 1} - Learning Rate: {current_lr}')

    print(f'Finished Training Model {m_idx + 1}. Min Val Log Loss: {model_scores[m_idx]:.2f}%')

print('Finished Training')

## Submission

In [ ]:
# Normalize weights
# Use validation log loss as the 'score'—lower is better, so we use its inverse
inverse_scores = [1.0 / score for score in model_scores]
total_inverse = sum(inverse_scores)
weights = [s / total_inverse for s in inverse_scores]

if test_loader is not None:
    # Match the LMSYS competition format for submission
    submission_data = {'id': [], 'winner_model_a': [], 'winner_model_b': [], 'winner_tie': []}
    
    for m in models: 
        m.eval()
    
    with torch.no_grad():
        for i, data in enumerate(test_loader):
            input_ids = data['input_ids'].to(DEVICE)
            attention_mask = data['attention_mask'].to(DEVICE)
            input_ids_swapped = data['input_ids_swapped'].to(DEVICE)
            attention_mask_swapped = data['attention_mask_swapped'].to(DEVICE)
            
            # Initialize ensemble predictions for this batch
            batch_size = input_ids.size(0)
            ensemble_probs = torch.zeros((batch_size, 3)).to(DEVICE)
            
            # Weighted Average of Probabilities (Soft Voting)
            # It is generally more stable to average probabilities rather than raw logits
            for m_idx, m in enumerate(models):
                # Pass 1: Original order
                logits_1 = m(input_ids, attention_mask)
                probs_1 = torch.softmax(logits_1, dim=1)
                
                # Pass 2: Swapped Order
                logits_2 = m(input_ids_swapped, attention_mask_swapped)
                probs_2 = torch.softmax(logits_2, dim=1)
                
                # Average TTA results for this model
                # Index 0=A, 1=B, 2=Tie. Flip [1, 0, 2] to align B/A back to A/B.
                model_avg_probs = (probs_1 + probs_2[:, [1, 0, 2]]) / 2
                
                # Add weighted contribution to ensemble
                ensemble_probs += model_avg_probs * weights[m_idx]
                
                # logits = m(input_ids, attention_mask)
                # probs = torch.softmax(logits, dim=1)
                # ensemble_probs += probs * weights[m_idx]
            
            # Move to CPU for pandas
            ensemble_probs_np = ensemble_probs.cpu().numpy()

            start_idx = i * test_loader.batch_size
            end_idx = start_idx + input_ids.size(0)
            batch_ids = base_test_df['id'].iloc[start_idx:end_idx].values

            for j, row_id in enumerate(batch_ids):
                submission_data['id'].append(row_id)
                submission_data['winner_model_a'].append(ensemble_probs_np[j, 0])
                submission_data['winner_model_b'].append(ensemble_probs_np[j, 1])
                submission_data['winner_tie'].append(ensemble_probs_np[j, 2])
    
    # Save to CSV
    submission_df = pd.DataFrame(submission_data)
    submission_df.to_csv('submission.csv', index=False)
    print(f"Submission file saved with {len(submission_df)} rows!")

### Verification

In [ ]:
if submission_df is not None:
    # 1. Check for missing values
    missing_values = submission_df.isnull().sum().sum()
    
    # 2. Check for correct column names
    expected_cols = ['id', 'winner_model_a', 'winner_model_b', 'winner_tie']
    missing_cols = [c for c in expected_cols if c not in submission_df.columns]
    
    # 3. Check for valid probability range [0, 1]
    prob_cols = ['winner_model_a', 'winner_model_b', 'winner_tie']
    out_of_range = submission_df[(submission_df[prob_cols] < 0).any(axis=1) | 
                                 (submission_df[prob_cols] > 1).any(axis=1)]
    
    # 4. Check if probabilities sum to ~1.0
    # We use a small epsilon (1e-5) to account for floating point rounding
    sums = submission_df[prob_cols].sum(axis=1)
    invalid_sums = submission_df[~sums.between(0.99, 1.01)]
    
    # 5. Final Verification Report
    print("--- LMSYS Submission Verification ---")
    print(f"Total Rows: {len(submission_df)}")
    print(f"Missing Values: {missing_values}")
    print(f"Missing Required Columns: {len(missing_cols)} ({missing_cols})")
    print(f"Invalid Probabilities (out of 0-1 range): {len(out_of_range)}")
    print(f"Rows where Probs don't sum to 1: {len(invalid_sums)}")
    
    # Validation Logic
    is_count_correct = (len(submission_df) == len(base_test_df))
    is_complete = (missing_values == 0 and len(missing_cols) == 0)
    is_valid = (len(out_of_range) == 0 and len(invalid_sums) == 0)
    
    if is_count_correct and is_complete and is_valid:
        print("\n✅ Verification Passed! Your file is ready for LMSYS submission.")
    else:
        print("\n❌ Verification Failed. Please check the counts above.")
        if not is_count_correct:
            print(f"   - Row count mismatch: Expected {len(base_test_df)}, got {len(submission_df)}")